# xbrl-extraction - Playground

This notebook walks through the full public surface of the `xbrl_extraction` module.
Each section builds on the previous one. Run the **Imports** cell first, then work through the sections in order.

## 0 - Imports and paths

In [2]:
from pathlib import Path

from xbrl_extraction import (
    Document,
    # Entry points
    extract,
)

# Convenience paths - adjust if you add more filings
INPUT_ZIP  = Path("../data/input/rawdata_us_1860_1860_XBRL_2025-11-10.zip")
FACTS_JSON = Path(r"..\data\output\rawdata_us_1860_1860_XBRL_2025-11-10.json")

---
## 1 - Extraction from the source ZIP

`extract(zip_path)` parses the iXBRL filing end-to-end and returns an `ExtractionResult`.
The result bundles the `Document` (facts + schema) and all four linkbase objects together.

Fields on `ExtractionResult`:
- `.document` -> `Document` (facts, periods, units, filing metadata)
- `.calc` -> `Calculations`
- `.pres` -> `Presentation`
- `.labs` -> `Labels`
- `.defs` -> `Definitions`

> **Try:** call `extract(INPUT_ZIP)`, inspect `result.document.filing`, count facts.

In [3]:
result = extract(INPUT_ZIP)
print(result.document.filing)
print(f'Filing contains {len(result.document.facts)} facts.')

Filing(form='10-K', fiscal_year=2025, fiscal_period='FY', period_end='2025-09-27', accounting_standard='US-GAAP', source_file='rawdata_us_1860_1860_XBRL_2025-11-10.zip', primary_document='aapl-20250927.htm')
Filing contains 830 facts.


---
## 2 - Loading from pre-extracted JSON

If you have already run the CLI and have the `data/output/` files, you do not need to re-extract.
Two loading strategies:

### 2a - `Document.load(path)` - facts only
Loads the `.facts.json`. Linkbases are `None` until you attach them manually with `doc.attach_calc(path)`, `doc.attach_pres(path)`, etc. Good when you only need facts.

### 2b - `Document.load_all(path)` - facts + all siblings auto-attached
Loads the `.facts.json` and automatically discovers and attaches any sibling `.calc.json` / `.pres.json` / `.labs.json` / `.defs.json` in the same directory.

> **Try:** load with `Document.load_all(FACTS_JSON)`, then call `print(doc.summary())` to confirm all four linkbases are attached.

In [4]:
# Your code here
doc = Document.load(FACTS_JSON)
print(doc.facts)

[Fact(concept='aapl:AccruedDistributionAndMarketingCurrent', value=8919.0, unit='usd', period='c-20', decimals='-6', scale='6', dimensions={}), Fact(concept='aapl:AccruedDistributionAndMarketingCurrent', value=7679.0, unit='usd', period='c-21', decimals='-6', scale='6', dimensions={}), Fact(concept='aapl:CashCashEquivalentsAndMarketableSecurities', value=132420.0, unit='usd', period='c-20', decimals='-6', scale='6', dimensions={}), Fact(concept='aapl:CashCashEquivalentsAndMarketableSecurities', value=156650.0, unit='usd', period='c-21', decimals='-6', scale='6', dimensions={}), Fact(concept='aapl:CashCashEquivalentsAndMarketableSecuritiesCost', value=134711.0, unit='usd', period='c-20', decimals='-6', scale='6', dimensions={}), Fact(concept='aapl:CashCashEquivalentsAndMarketableSecuritiesCost', value=160600.0, unit='usd', period='c-21', decimals='-6', scale='6', dimensions={}), Fact(concept='aapl:CashEquivalentsAndMarketableSecuritiesAccumulatedGrossUnrealizedGainBeforeTax', value=736.

In [26]:
doc = Document.load(r'C:\Users\AugustinHourquet\Documents\xbrl-extraction\data\output\rawdata_us_13091_13091_XBRL_2026-02-26.facts.json')
doc.attach_calc(r'C:\Users\AugustinHourquet\Documents\xbrl-extraction\data\output\rawdata_us_13091_13091_XBRL_2026-02-26.calc.json')
print(doc.summary())

10-K — FY2025 (US-GAAP)
  Source:    rawdata_us_13091_13091_XBRL_2026-02-26.zip
  Primary:   ande-20251231.htm
  Period end: 2025-12-31

Facts:  1264
  By namespace:
    us-gaap:                     1150
    ande:                        112
    dei:                         2
  By period type:
    duration                     693
    instant                      571
  Dimensional facts:           621 (49%)
  Top axes:
    us-gaap:StatementEquityComponentsAxis    97
    srt:ConsolidationItemsAxis               97
    us-gaap:DerivativeInstrumentRiskAxis     94
    us-gaap:StatementBusinessSegmentsAxis    83
    us-gaap:DebtInstrumentAxis               65

Attached linkbases:
  ✓ calc  (240 arcs across 23 roles)
  ✗ pres  (not attached)
  ✗ labs  (not attached)
  ✗ defs  (not attached)



In [27]:
doc = Document.load_all(FACTS_JSON)
print(doc.summary())

10-K — FY2025 (US-GAAP)
  Source:    rawdata_us_13091_13091_XBRL_2026-02-26.zip
  Primary:   ande-20251231.htm
  Period end: 2025-12-31

Facts:  1264
  By namespace:
    us-gaap:                     1150
    ande:                        112
    dei:                         2
  By period type:
    duration                     693
    instant                      571
  Dimensional facts:           621 (49%)
  Top axes:
    us-gaap:StatementEquityComponentsAxis    97
    srt:ConsolidationItemsAxis               97
    us-gaap:DerivativeInstrumentRiskAxis     94
    us-gaap:StatementBusinessSegmentsAxis    83
    us-gaap:DebtInstrumentAxis               65

Attached linkbases:
  ✓ calc  (240 arcs across 23 roles)
  ✓ pres  (1181 arcs, 105 statements)
  ✓ labs  (2062 labels)
  ✓ defs  (733 arcs)



---
## 3 - Exploring the Document

Once you have a `doc`, the core attributes are:

| Attribute | Type | What it holds |
|---|---|---|
| `doc.filing` | `Filing` | form, fiscal_year, period_end, accounting_standard, ... |
| `doc.facts` | `list[Fact]` | every numeric fact |
| `doc.periods` | `dict[str, Period]` | context_id -> Period (instant or duration) |
| `doc.units` | `dict[str, Unit]` | unit_id -> Unit (measure or numerator/denominator) |

Each `Fact` has: `.concept`, `.value`, `.unit`, `.period`, `.scale`, `.decimals`, `.dimensions`.

> **Try:** peek at `doc.facts[:5]`, inspect a single fact period via `doc.periods[fact.period]`, check `doc.filing.accounting_standard`.

In [ ]:
# Your code here
doc = Document.load_all(FACTS_JSON)
doc.units, doc.periods

({'usd': Unit(measure='iso4217:USD', numerator=None, denominator=None),
  't': Unit(measure='utr:T', numerator=None, denominator=None),
  'usdPerShare': Unit(measure=None, numerator='iso4217:USD', denominator='xbrli:shares'),
  'number': Unit(measure='xbrli:pure', numerator=None, denominator=None),
  'gal': Unit(measure='utr:gal', numerator=None, denominator=None),
  'shares': Unit(measure='xbrli:shares', numerator=None, denominator=None)},
 {'c-148': Period(type='instant', date='2024-12-31', start=None, end=None),
  'c-329': Period(type='duration', date=None, start='2025-01-01', end='2025-12-31'),
  'c-258': Period(type='duration', date=None, start='2025-01-01', end='2025-12-31'),
  'c-255': Period(type='duration', date=None, start='2025-01-01', end='2025-12-31'),
  'c-355': Period(type='instant', date='2024-12-31', start=None, end=None),
  'c-41': Period(type='duration', date=None, start='2025-01-01', end='2025-12-31'),
  'c-177': Period(type='instant', date='2025-12-31', start=None,

In [31]:
fact = doc.facts[1]
fact, doc.periods[fact.period], doc.filing.accounting_standard

(Fact(concept='ande:AccountsPayableTradeandOtherCurrent', value=1047436.0, unit='usd', period='c-7', decimals='-3', scale='3', dimensions={}),
 Period(type='instant', date='2024-12-31', start=None, end=None),
 'US-GAAP')

---
## 4 - Filtering facts

`doc.filter(**kwargs)` returns a **new** `Document` containing only facts that match every criterion. Filters AND together. The returned doc has its `periods` and `units` pruned to what is still referenced.

Available filter kwargs:

| kwarg | Description |
|---|---|
| `concept` | exact concept name, e.g. `"us-gaap:Assets"` |
| `concept_contains` | substring match (case-insensitive) |
| `concept_in` | list/set of concept names |
| `period` | exact context_id key |
| `period_end` | ISO date - matches any context ending on that date |
| `period_type` | `"instant"` or `"duration"` |
| `fiscal_year_only` | `True` - keeps only the full-year duration context |
| `unit` | exact unit_id |
| `currency` | `True` - keeps only `iso4217:` units |
| `no_dimensions` | `True` - drops all dimensional facts |
| `has_axis` | axis concept name must be present in dimensions |
| `has_member` | member concept name must be a value in dimensions |
| `statement` | `role_short` string (requires pres attached) |
| `statement_contains` | substring match on role_definition (requires pres) |
| `accounting_standard` | `"US-GAAP"` or `"IFRS"` |

> **Try:** filter to a specific concept, to duration facts only, or to a statement. Chain filters: `doc.filter(currency=True).filter(no_dimensions=True)`.

In [34]:
# Your code here
doc = Document.load_all(FACTS_JSON)
doc.filter(statement_contains = 'BALANCE')

Document(filing=Filing(form='10-K', fiscal_year=2025, fiscal_period='FY', period_end='2025-12-31', accounting_standard='US-GAAP', source_file='rawdata_us_13091_13091_XBRL_2026-02-26.zip', primary_document='ande-20251231.htm'), periods={'c-251': Period(type='instant', date='2024-12-31', start=None, end=None), 'c-233': Period(type='instant', date='2025-12-31', start=None, end=None), 'c-247': Period(type='duration', date=None, start='2025-01-01', end='2025-12-31'), 'c-333': Period(type='instant', date='2023-12-31', start=None, end=None), 'c-277': Period(type='instant', date='2024-12-31', start=None, end=None), 'c-6': Period(type='instant', date='2025-12-31', start=None, end=None), 'c-36': Period(type='instant', date='2024-12-31', start=None, end=None), 'c-295': Period(type='instant', date='2024-12-31', start=None, end=None), 'c-274': Period(type='instant', date='2024-12-31', start=None, end=None), 'c-7': Period(type='instant', date='2024-12-31', start=None, end=None), 'c-47': Period(type=

---
## 5 - Exporting to a DataFrame

`doc.to_dataframe()` flattens all facts into a long-format pandas DataFrame. Periods and units are denormalised into per-row columns. If labels are attached, a `label` column is added.

Resulting columns: `concept`, `value`, `unit`, `unit_measure`, `period`, `period_type`, `period_start`, `period_end`, `period_date`, `scale`, `decimals`, `dimensions`, `source_file`, (`label` if labs attached).

> **Try:** call `doc.to_dataframe()`, then filter/group in pandas - e.g. group by `concept` to see all reported values for a given line item across periods.

In [ ]:
# Your code here
doc = Document.load_all(FACTS_JSON)
df = doc.filter(statement_contains = "BALANCE").to_dataframe()
df.groupby("unit").first()


,concept,value,unit_measure,period,period_type,period_start,period_end,period_date,scale,decimals,dimensions,source_file,label
unit,,,,,,,,,,,,,
shares,us-gaap:CommonStockSharesAuthorized,63000.0,xbrli:shares,c-6,instant,NaN,NaN,2025-12-31,3,INF,None,rawdata_us_13091_13091_XBRL_2026-02-26.zip,"Common Stock, Shares Authorized"
usd,ande:AccountsPayableTradeandOtherCurrent,918691.0,iso4217:USD,c-6,instant,2025-01-01,2025-12-31,2025-12-31,3,-3,{'us-gaap:FairValueByAssetClassAxis': 'us-gaap...,rawdata_us_13091_13091_XBRL_2026-02-26.zip,"Accounts Payable, Trade and Other, Current"


---
## 6 - Calc linkbase - graph navigation

Requires `doc.calc` to be attached (it is after `load_all`).

### Available statements
Get the list of calc roles from `doc.calc.role_definitions` (a `dict[role_short, label]`).

### Navigation methods

| Method | Returns | Description |
|---|---|---|
| `doc.children_of(concept, role_short=None)` | `list[dict]` | Direct children in the calc graph. Each dict: `concept`, `weight`, `order`, `role`. |
| `doc.parent_of(concept, role_short=None)` | `list[dict]` | Direct parents. Each dict: `concept`, `weight`, `role`. |
| `doc.tree(role_short)` | `dict` | Full nested dict of the calc tree for one role. Root -> children -> grandchildren. |
| `doc.expand(concept, role_short, period_end)` | `DataFrame` | Walk a subtree and join actual fact values. Columns: `concept`, `depth`, `weight_path`, `value`, `label`. |

> **Try:** list `doc.calc.role_definitions`, pick a role, call `doc.tree(role_short)`, then `doc.expand(root_concept, role_short, doc.filing.period_end)`.

In [ ]:
# Your code here
doc = Document.load_all(FACTS_JSON)

for role in doc.calc.role_definitions:
    if doc.calc.role_definitions[role] == "Property, Plant and Equipment":
        print(f'{role}: {doc.calc.role_definitions[role]}' )

doc.children_of(concept = 'us-gaap:Assets'), doc.parent_of('us-gaap:PropertyPlantAndEquipmentNet')

http://www.andersonsinc.com/role/Cover: Cover
http://www.andersonsinc.com/role/AuditInformation: Audit Information
http://www.andersonsinc.com/role/ConsolidatedStatementsofOperations: Consolidated Statements of Operations
http://www.andersonsinc.com/role/ConsolidatedStatementsofComprehensiveIncome: Consolidated Statements of Comprehensive Income
http://www.andersonsinc.com/role/ConsolidatedBalanceSheets: Consolidated Balance Sheets
http://www.andersonsinc.com/role/ConsolidatedBalanceSheetsParenthetical: Consolidated Balance Sheets (Parenthetical)
http://www.andersonsinc.com/role/ConsolidatedStatementsofCashFlows: Consolidated Statements of Cash Flows
http://www.andersonsinc.com/role/ConsolidatedStatementsofEquity: Consolidated Statements of Equity
http://www.andersonsinc.com/role/ConsolidatedStatementsofEquityParenthetical: Consolidated Statements of Equity (Parenthetical)
http://www.andersonsinc.com/role/SummaryofSignificantAccountingPolicies: Summary of Significant Accounting Policie

([{'concept': 'us-gaap:InvestmentsAndOtherNoncurrentAssets',
   'weight': 1.0,
   'order': 1.0,
   'role': 'ConsolidatedBalanceSheets'},
  {'concept': 'us-gaap:PropertyPlantAndEquipmentNet',
   'weight': 1.0,
   'order': 2.0,
   'role': 'ConsolidatedBalanceSheets'},
  {'concept': 'us-gaap:AssetsCurrent',
   'weight': 1.0,
   'order': 3.0,
   'role': 'ConsolidatedBalanceSheets'}],
 [{'concept': 'us-gaap:Assets',
   'weight': 1.0,
   'role': 'ConsolidatedBalanceSheets'}])

In [68]:
doc = Document.load_all(FACTS_JSON)
doc.tree('ConsolidatedBalanceSheets')

{'us-gaap:Assets': {'us-gaap:InvestmentsAndOtherNoncurrentAssets': {'us-gaap:OtherIntangibleAssetsNet': {},
   'us-gaap:OtherAssetsNoncurrent': {},
   'us-gaap:OperatingLeaseRightOfUseAsset': {},
   'us-gaap:Goodwill': {}},
  'us-gaap:PropertyPlantAndEquipmentNet': {},
  'us-gaap:AssetsCurrent': {'us-gaap:CashAndCashEquivalentsAtCarryingValue': {},
   'us-gaap:AccountsNotesAndLoansReceivableNetCurrent': {},
   'us-gaap:InventoryNet': {},
   'us-gaap:CommodityContractAssetCurrent': {},
   'us-gaap:OtherAssetsCurrent': {}}},
 'us-gaap:LiabilitiesAndStockholdersEquity': {'us-gaap:Liabilities': {'us-gaap:LiabilitiesCurrent': {'us-gaap:LinesOfCreditCurrent': {},
    'ande:AccountsPayableTradeandOtherCurrent': {},
    'us-gaap:ContractWithCustomerLiabilityCurrent': {},
    'us-gaap:DerivativeLiabilitiesCurrent': {},
    'us-gaap:AccruedLiabilitiesCurrent': {},
    'us-gaap:LongTermDebtCurrent': {}},
   'us-gaap:LongTermDebtNoncurrent': {},
   'us-gaap:OperatingLeaseLiabilityNoncurrent': {},


In [70]:
doc = Document.load_all(FACTS_JSON)
doc.expand(concept = 'us-gaap:Assets',
           role_short = 'ConsolidatedBalanceSheets',
           period_end = doc.filing.period_end)

,concept,depth,weight_path,value,label
0,us-gaap:Assets,0,1.0,3712832.0,Assets
1,us-gaap:InvestmentsAndOtherNoncurrentAssets,1,1.0,396923.0,Investments and Other Noncurrent Assets
2,us-gaap:OtherIntangibleAssetsNet,2,1.0,63510.0,"Other Intangible Assets, Net"
3,us-gaap:OtherAssetsNoncurrent,2,1.0,96765.0,"Other Assets, Noncurrent"
4,us-gaap:OperatingLeaseRightOfUseAsset,2,1.0,108792.0,"Operating Lease, Right-of-Use Asset"
5,us-gaap:Goodwill,2,1.0,127856.0,Goodwill
6,us-gaap:PropertyPlantAndEquipmentNet,1,1.0,939500.0,"Property, Plant and Equipment, Net"
7,us-gaap:AssetsCurrent,1,1.0,2376409.0,"Assets, Current"
8,us-gaap:CashAndCashEquivalentsAtCarryingValue,2,1.0,98283.0,Cash and Cash Equivalent
9,us-gaap:AccountsNotesAndLoansReceivableNetCurrent,2,1.0,652472.0,"Accounts and Financing Receivable, after Allow..."


---
## 7 - Calc verification

`doc.verify(role_short=None, period_end=None, tolerance=0.0)` checks whether the reported subtotals are arithmetically consistent with their children.

Returns a DataFrame with columns: `parent`, `expected`, `actual`, `diff`, `status`, `role`.

Status values:
- `"match"` - exact (or within tolerance)
- `"rounding"` - small diff <= tolerance (only when tolerance > 0)
- `"mismatch"` - diff > tolerance
- `"missing_children"` - at least one child fact is absent for this period
- `"missing_parent"` - parent fact absent (expected is still computed)

Defaults to `doc.filing.period_end` if `period_end` is not passed. Default tolerance is **strict** (0.0).

> **Try:** `doc.verify()` to see all roles, then narrow with `role_short=`. Re-run with `tolerance=1.0` to suppress filer rounding.

In [71]:
# Your code here
doc = Document.load_all(FACTS_JSON)
doc.verify()


,parent,expected,actual,diff,status,role
0,us-gaap:OtherComprehensiveIncomeLossNetOfTax,NaN,-1248.0,NaN,missing_children,AccumulatedOtherComprehensiveIncomeLossDetails
1,us-gaap:BusinessCombinationConsiderationTransf...,NaN,NaN,NaN,missing_children,BusinessAcquisitionPurchasePriceAllocationDetails
2,us-gaap:BusinessCombinationRecognizedIdentifia...,NaN,NaN,NaN,missing_children,BusinessAcquisitionPurchasePriceAllocationDetails
3,us-gaap:BusinessCombinationRecognizedIdentifia...,NaN,NaN,NaN,missing_children,BusinessAcquisitionPurchasePriceAllocationDetails
4,us-gaap:BusinessCombinationRecognizedIdentifia...,NaN,NaN,NaN,missing_children,BusinessAcquisitionPurchasePriceAllocationDetails
5,us-gaap:Assets,3712832.0,3712832.0,0.0,match,ConsolidatedBalanceSheets
6,us-gaap:AssetsCurrent,2376409.0,2376409.0,0.0,match,ConsolidatedBalanceSheets
7,us-gaap:InvestmentsAndOtherNoncurrentAssets,396923.0,396923.0,0.0,match,ConsolidatedBalanceSheets
8,us-gaap:Liabilities,2422597.0,2422597.0,0.0,match,ConsolidatedBalanceSheets
9,us-gaap:LiabilitiesAndStockholdersEquity,NaN,3712832.0,NaN,missing_children,ConsolidatedBalanceSheets


---
## 8 - Rendering a statement

`doc.render_statement(role_short, period_end, print=False)` reconstructs a financial statement as indented text, using the **presentation** linkbase for structure and the **calc** linkbase for +/- signs. Labels are resolved if `labs` is attached.

Layout:
```
<Statement title> - period ending YYYY-MM-DD (in millions USD)

  <Section header>
    <Subtotal>                                       123,456
      + <Leaf concept>                                12,345
      - <Leaf concept>                               (12,345)
```

Pass `print=True` to write directly to stdout (cleaner in notebooks).

> **Try:** pick a `role_short` from `doc.pres.role_definitions`, then call `doc.render_statement(role_short, doc.filing.period_end, print=True)`.

In [7]:
# Your code here

doc.render_statement("CONSOLIDATEDBALANCESHEETS", doc.filing.period_end, True)

CONSOLIDATED BALANCE SHEETS — period ending 2025-09-27 (in millions USD)

    Statement of Financial Position [Abstract]
      ASSETS:
        Current assets:
        + Cash and cash equivalents                                  35,934
        + Marketable securities                                      18,763
        + Accounts receivable, net                                   39,777
        + Vendor non-trade receivables                               33,180
        + Inventories                                                 5,718
        + Other current assets                                       14,585
          Total current assets                                      147,957
        Non-current assets:
        + Marketable securities                                      77,723
        + Property, plant and equipment, net                         49,834
        + Other non-current assets                                   83,727
          Total non-current assets                   

'CONSOLIDATED BALANCE SHEETS — period ending 2025-09-27 (in millions USD)\n\n    Statement of Financial Position [Abstract]\n      ASSETS:\n        Current assets:\n        + Cash and cash equivalents                                  35,934\n        + Marketable securities                                      18,763\n        + Accounts receivable, net                                   39,777\n        + Vendor non-trade receivables                               33,180\n        + Inventories                                                 5,718\n        + Other current assets                                       14,585\n          Total current assets                                      147,957\n        Non-current assets:\n        + Marketable securities                                      77,723\n        + Property, plant and equipment, net                         49,834\n        + Other non-current assets                                   83,727\n          Total non-current assets  

---
## 9 - Linkbase containers (raw access)

Beyond the high-level methods, you can access the raw linkbase objects directly.

### Labels - `doc.labs`
- `doc.labs.entries` -> `list[LabelEntry]` (each has `.concept`, `.label`, `.role`, `.lang`)
- `doc.labs.get(concept, preferred_label=None)` -> `str | None` - resolves the best human-readable label for a concept

### Presentation - `doc.pres`
- `doc.pres.arcs` -> `list[PresArc]` (each has `.role`, `.role_short`, `.role_definition`, `.parent`, `.child`, `.order`, `.preferred_label`)
- `doc.pres.role_definitions` -> `dict[role_short, label]`

### Calculations - `doc.calc`
- `doc.calc.arcs` -> `list[CalcArc]` (each has `.role`, `.role_short`, `.parent`, `.child`, `.weight`, `.order`)
- `doc.calc.role_definitions` -> `dict[role_short, label]`

### Definitions - `doc.defs`
- `doc.defs.arcs` -> `list[DefArc]` (dimensional hypercube relationships)

> **Try:** print `doc.labs.get("us-gaap:Assets")`, list `doc.pres.role_definitions`, inspect a few raw arcs.

In [ ]:
# Your code here


---
## 10 - Round-trip serialisation

`doc.to_dict()` produces the JSON-compatible dict (facts payload only - linkbases excluded by design). `Document.from_dict(d)` is the inverse.

Useful for:
- Writing a filtered subset back to a new `.facts.json`
- Passing a Document to another process via JSON
- Debugging: confirm the dict round-trips cleanly

> **Try:** `d = doc.to_dict()`, then `doc2 = Document.from_dict(d)` and verify `len(doc2.facts) == len(doc.facts)`.

In [ ]:
# Your code here
